In [3]:
import os
import cv2

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from src.utils import HandDetection, PoseDetection, FaceDetection

# =========================
# Input Video
# =========================

input_video = r"D:\SignDetection\datasets\raw\WLASL\videos\18323.mp4"

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {input_video}")

# =========================
# Detection
# =========================

hand_detection = HandDetection(
    min_hand_detection_confidence=0.2
)

pose_detection = PoseDetection(
    min_pose_detection_confidence=0.2
)

# =========================
# Video Info
# =========================

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {input_video}")
print(f"Width: {width}, Height: {height}")
print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print("Video started.")
print("Press 'q' to quit.")

# =========================
# Main Loop
# =========================

frame_index = 0

while True:

    success, frame = cap.read()

    if not success:
        print("Video finished.")
        break

    # =========================
    # BGR -> RGB
    # =========================

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # =========================
    # MediaPipe video timestamp
    # =========================

    timestamp_ms = int(
        frame_index * 1000 / fps
    )

    # =========================
    # Hand Detection
    # =========================

    detection_hand_results = hand_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    # =========================
    # Pose Detection
    # =========================

    detection_pose_results = pose_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    # =========================
    # Draw Landmarks
    # =========================

    output = hand_detection.draw_landmarks_on_image(
        rgb_frame.copy(),
        detection_hand_results
    )

    output = pose_detection.draw_landmarks_on_image(
        output,
        detection_pose_results
    )

    # =========================
    # RGB -> BGR
    # =========================

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    # =========================
    # Display
    # =========================

    cv2.imshow(
        "Video - Landmarks + Lips",
        output
    )

    frame_index += 1

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("x"):
        break

# =========================
# Release
# =========================

cap.release()
cv2.destroyAllWindows()

hand_detection.close()

print("Video stopped.")

Video: D:\SignDetection\datasets\raw\WLASL\videos\18323.mp4
Width: 320, Height: 240
FPS: 25.0
Total frames: 87
Video started.
Press 'q' to quit.
Video finished.
Video stopped.
